# 05 Evaluation
Evaluation of the best model of each architecture (selected on the validation set) on the test set, comparison with the expectations in EXPECTATIONS.md, and examples the models got wrong. The hypothesis-only runs of both models are also evaluated, as diagnostics. Per-choice scoring of model 1 is evaluated as a separate comparison.

In [ ]:
# setup: on Colab, connect Google Drive, where the features and checkpoints are saved
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/nlp-project1-art")
    %pip install -q datasets

## Chosen models
The test set is evaluated only in this notebook, once, after both models were chosen on the validation set (`01_data_exploration.ipynb` only counts its labels, and `02_preprocessing.ipynb` only checks its texts and prepares its input; no model was trained, tuned or selected on it): "Test it on the test data for the final comparison. Do not tune hyperparameters on the test set!" (course slides "Course Project HS26", slide 10). The primary comparison uses the fixed-position input format. For each architecture, the run and checkpoint are selected by validation loss from the grid and sweep experiments in notebooks 03 and 04. The variants in notebook 04b are additional experiments and do not replace these selected models. Per-choice scoring improves model 1 on validation and is also evaluated on the test set as a separate comparison.
- Model 1: 100 hidden units, learning rate 0.000316, batch size 32, best epoch 13 (validation loss 0.6445, accuracy 63.20%; `03_model1_embeddings.ipynb`, section "Sweep").
- Model 2: GRU, RNN hidden size 128, learning rate 0.001, dropout 0.5, best epoch 4 (validation loss 0.5872, accuracy 67.60%; `04_model2_rnn.ipynb`).

Diagnostics only, not candidates: the hypothesis-only runs of both models with their chosen configurations (`04b_preprocessing_experiments.ipynb`): model 1 with best epoch 12, validation loss 0.6347, accuracy 64.40%, and model 2 with best epoch 4, validation loss 0.6014, accuracy 67.70%. They show how much of their validation accuracy remains on the adversarially filtered test set.

Separate comparison, not a candidate: per-choice scoring of model 1, with the settings of the chosen model 1 (best epoch 32, validation loss 0.5829, accuracy 69.10%; `04b_preprocessing_experiments.ipynb`).

Nothing is chosen or changed because of any of these, nor because of the runs with other seeds (section "Test results with other seeds").

In [ ]:
# the chosen runs: name of the checkpoint in checkpoints/
model1_name = "glove-mlp-sweep-h100-lr0.000316-b32"
model2_name = "glove-gru-h128-lr0.001-d0.5"
# settings of the runs (needed to build the same models before loading the weights)
model1_inner_dim = 100
model2_rnn_type = "gru"
model2_dropout = 0.5
# diagnostic only, not a candidate: model 1 trained on h1 and h2 alone, with its chosen configuration
hyp_only_name = "prep-mlp-hypotheses-only"
hyp_only_inner_dim = 100
# diagnostic only, not a candidate: model 2 trained on h1 and h2 alone, with its chosen configuration
hyp_only2_name = "prep-gru-hypotheses-only"
# separate comparison, not a candidate: model 1 with per-choice scoring, with the settings of the chosen model 1
per_choice_name = "prep-mlp-per-choice"


## Test data
Load the test data of both models, saved in `02_preprocessing.ipynb`: the averaged GloVe vectors for model 1, and the token indices and the embedding matrix for model 2. Both files contain the same labels. The hypothesis-only runs use h1 and h2: the last 600 of the 1200 numbers for model 1, the last 2 of the 4 texts for model 2. The validation data is loaded too, to check that the loaded models are the chosen ones.

In [ ]:
import numpy as np
import torch

data1 = np.load("features/model1.npz")
data2 = np.load("features/model2.npz")
# model 1: average GloVe vectors of O1, O2, h1, h2 side by side, shape [examples, 1200]
X1_valid = torch.tensor(data1["X_valid"])
X1_test = torch.tensor(data1["X_test"])
# model 2: token indices, shape [examples, 4, 34], and the GloVe embedding matrix
X2_valid = torch.tensor(data2["X_valid"])
X2_test = torch.tensor(data2["X_test"])
embedding_matrix = torch.tensor(data2["embedding_matrix"])
# labels (0 = h1, 1 = h2): the same in both files
y_valid = torch.tensor(data1["y_valid"])
y_test = torch.tensor(data1["y_test"])
assert torch.equal(y_test, torch.tensor(data2["y_test"])), "the test labels of the two files differ"
print(X1_test.shape, X2_test.shape, y_test.shape)

## Models
The same classes as in `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb`, copied, because a notebook cannot use the classes defined in another notebook. `RNNClassifier` also has `num_texts` from `04b_preprocessing_experiments.ipynb`, because the hypothesis-only run of model 2 has 2 texts instead of 4. `PerChoiceClassifier` is copied from the same notebook, for the separate comparison.

In [ ]:
import torch.nn as nn

# same class as in 03_model1_embeddings.ipynb
class Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, inner_dim=10):
        super().__init__()
        self.linear1 = nn.Linear(input_dim, inner_dim)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(inner_dim, output_dim)

    def forward(self, x):
        out = self.linear1(x)
        out = self.relu(out)
        out = self.linear2(out)
        return out

from torch.nn.utils.rnn import pack_padded_sequence

class RNNClassifier(nn.Module):

    def __init__(self, embedding_matrix, rnn_type="lstm", hidden_dim=128, num_layers=2, dropout=0.0, inner_dim=100, num_texts=4):
        super().__init__()
        # embedding layer: the frozen GloVe vectors, row 0 = padding
        self.embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=True, padding_idx=0)
        # size of one word vector (300)
        embedding_dim = embedding_matrix.shape[1]
        # 2-layer LSTM or GRU; batch_first=True: input shape [batch size, sequence length, embedding dim]
        if rnn_type == "lstm":
            self.rnn = nn.LSTM(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        else:
            self.rnn = nn.GRU(embedding_dim, hidden_dim, num_layers=num_layers, dropout=dropout, batch_first=True)
        # dropout on the final hidden states before the classifier (0.0 = no dropout)
        self.dropout = nn.Dropout(dropout)
        # classifier on the final hidden states side by side (O1, O2, h1, h2, or only h1, h2)
        self.classifier = Classifier(num_texts * hidden_dim, 2, inner_dim=inner_dim)
        self.num_texts = num_texts

    def forward(self, x):
        # x: token indices, shape [batch size, number of texts, 34]
        features = []
        for i in range(self.num_texts):
            # one text (O1, O2, h1 or h2) of every example in the batch: [batch size, 34]
            text = x[:, i]
            # real length of each text = number of token indices that are not 0 (padding)
            lengths = (text != 0).sum(dim=1)
            # look up the GloVe vectors: [batch size, 34, 300]
            embedded = self.embedding(text)
            # give the RNN the real lengths, so it stops after the last real word
            packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
            # the RNN reads the text; only the final hidden state is needed
            _, hidden = self.rnn(packed)
            # the LSTM returns (h_n, c_n), the GRU only h_n
            if isinstance(self.rnn, nn.LSTM):
                hidden = hidden[0]
            # h_n of the last layer: [batch size, hidden dim]
            features.append(hidden[-1])
        # the final hidden states side by side: [batch size, number of texts * hidden dim]
        features = torch.concat(features, dim=-1)
        # dropout on the final hidden states (active only in training mode)
        features = self.dropout(features)
        return self.classifier(features)

# same class as in 04b_preprocessing_experiments.ipynb
# one score per hypothesis: the same classifier scores each hypothesis
class PerChoiceClassifier(nn.Module):

    def __init__(self, text_dim, inner_dim, num_observations=2):
        super().__init__()
        # size of the vector of one text (model 1: 300, model 2: RNN hidden size)
        self.text_dim = text_dim
        # how many observation vectors come before h1 and h2 (0 = none)
        self.num_observations = num_observations
        # input: the observations and one hypothesis; output: 1 score
        self.scorer = Classifier((num_observations + 1) * text_dim, 1, inner_dim=inner_dim)

    def forward(self, x):
        # x side by side: the observations, h1, h2
        start = self.num_observations * self.text_dim
        observations = x[:, :start]
        h1 = x[:, start:start + self.text_dim]
        h2 = x[:, start + self.text_dim:]
        # the same scorer for both hypotheses: [batch size, 1] each
        score1 = self.scorer(torch.concat([observations, h1], dim=1))
        score2 = self.scorer(torch.concat([observations, h2], dim=1))
        # the two scores are the 2 logits: [batch size, 2], as in the baseline
        return torch.concat([score1, score2], dim=1)

## Evaluation function
The same `evaluate()` as in the training notebooks, except that it also returns the predictions, so that the accuracy, the comparisons and the error analysis all use the same predictions (on near-tied examples, the prediction can depend on the batch size; see "Result for model 1" in `04b_preprocessing_experiments.ipynb`). All batches have the same size, so that the average of the batch losses is the average over all examples: the test set has 1532 = 4 x 383 examples, so 4 batches of 383; the validation set 10 batches of 100, as before.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# cross-entropy loss, as in training
criterion = nn.CrossEntropyLoss()

# same function as in 03_model1_embeddings.ipynb and 04_model2_rnn.ipynb, and it also returns the predictions
def evaluate(model, loader):
    # evaluation mode, and no gradients are needed
    model.eval()
    losses, predictions, labels = [], [], []
    with torch.no_grad():
        for x, y in loader:
            output = model(x)
            # loss of this batch
            losses.append(criterion(output, y).item())
            # predicted class: the hypothesis with the higher logit
            predictions.append(output.argmax(dim=1))
            labels.append(y)
    # join the batches: one prediction and one label per example
    predictions = torch.cat(predictions)
    labels = torch.cat(labels)
    # share of correct predictions
    accuracy = (predictions == labels).float().mean().item()
    return np.mean(losses), accuracy, predictions

# equal batches: validation 10 x 100, test 4 x 383
valid_loader1 = DataLoader(TensorDataset(X1_valid, y_valid), batch_size=100)
test_loader1 = DataLoader(TensorDataset(X1_test, y_test), batch_size=383)
valid_loader2 = DataLoader(TensorDataset(X2_valid, y_valid), batch_size=100)
test_loader2 = DataLoader(TensorDataset(X2_test, y_test), batch_size=383)
# hypothesis-only diagnostic: only h1 and h2, the last 600 of the 1200 numbers, same batches
valid_loader_hyp = DataLoader(TensorDataset(X1_valid[:, 600:], y_valid), batch_size=100)
test_loader_hyp = DataLoader(TensorDataset(X1_test[:, 600:], y_test), batch_size=383)
# hypothesis-only diagnostic of model 2: only h1 and h2, the last 2 of the 4 texts, same batches
valid_loader_hyp2 = DataLoader(TensorDataset(X2_valid[:, 2:], y_valid), batch_size=100)
test_loader_hyp2 = DataLoader(TensorDataset(X2_test[:, 2:], y_test), batch_size=383)
print(len(valid_loader1), len(test_loader1))

## Loading the chosen models
Build each model with the settings of its run and load the weights of the best epoch. Check on the validation set first: the loss and accuracy must be exactly those of the best epoch in the training notebooks.

In [ ]:
# model 1: 1200 inputs, 2 outputs, hidden units of its run
model1 = Classifier(1200, 2, inner_dim=model1_inner_dim)
model1.load_state_dict(torch.load(f"checkpoints/{model1_name}.pth", map_location="cpu", weights_only=True))
# model 2: same settings as in its training run
model2 = RNNClassifier(embedding_matrix, rnn_type=model2_rnn_type, dropout=model2_dropout)
model2.load_state_dict(torch.load(f"checkpoints/{model2_name}.pth", map_location="cpu", weights_only=True))

# hypothesis-only diagnostic: 600 inputs (h1 and h2)
model_hyp = Classifier(600, 2, inner_dim=hyp_only_inner_dim)
model_hyp.load_state_dict(torch.load(f"checkpoints/{hyp_only_name}.pth", map_location="cpu", weights_only=True))
# hypothesis-only diagnostic of model 2: 2 texts (h1 and h2), settings of the chosen model 2
model_hyp2 = RNNClassifier(embedding_matrix, rnn_type=model2_rnn_type, dropout=model2_dropout, num_texts=2)
hyp2_weights = torch.load(f"checkpoints/{hyp_only2_name}.pth", map_location="cpu", weights_only=True)
# 04b does not load the test set, so this embedding has no rows for the test words; it is frozen, so the saved matrix replaces it
hyp2_weights["embedding.weight"] = embedding_matrix
model_hyp2.load_state_dict(hyp2_weights)
# separate comparison: per-choice scoring of model 1, the same 1200 numbers and hidden units
model_pc = PerChoiceClassifier(300, inner_dim=model1_inner_dim)
model_pc.load_state_dict(torch.load(f"checkpoints/{per_choice_name}.pth", map_location="cpu", weights_only=True))

for name, model, loader in [(model1_name, model1, valid_loader1), (model2_name, model2, valid_loader2), (hyp_only_name, model_hyp, valid_loader_hyp), (hyp_only2_name, model_hyp2, valid_loader_hyp2), (per_choice_name, model_pc, valid_loader1)]:
    valid_loss, valid_acc, _ = evaluate(model, loader)
    print(f"{name}: validation loss {valid_loss:.4f}, accuracy {valid_acc:.2%}")

## Test results
The one evaluation of both models on the test set (the primary comparison), of the hypothesis-only runs as diagnostics, and of per-choice scoring of model 1 as a separate comparison.

**Test evaluation pending:** this notebook has not been run yet.

In [ ]:
# predictions of each run, for the comparisons and the error analysis below
test_predictions = {}
for name, model, loader in [(model1_name, model1, test_loader1), (model2_name, model2, test_loader2), (hyp_only_name, model_hyp, test_loader_hyp), (hyp_only2_name, model_hyp2, test_loader_hyp2)]:
    test_loss, test_acc, test_predictions[name] = evaluate(model, loader)
    print(f"{name}: test loss {test_loss:.4f}, accuracy {test_acc:.2%}")

# separate comparison: per-choice scoring of model 1, with the test batches of model 1
test_loss, test_acc, test_predictions[per_choice_name] = evaluate(model_pc, test_loader1)
print(f"separate comparison, {per_choice_name}: test loss {test_loss:.4f}, accuracy {test_acc:.2%}")

## Test results with other seeds
The chosen configurations were also trained with seeds 1 to 4 (`03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb`, sections "Other seeds"). All five seeds are evaluated once on the test set, to report how much the test accuracy changes with the seed (course slides "Project Discussions" (Week 7), slide 42). This is only a report: seed 42 stays the chosen model, and nothing is chosen or changed because of these results.

In [ ]:
# the chosen configurations with all five seeds; seed 42 stays the chosen model
seed_runs = {
    "model 1": [model1_name] + [f"glove-mlp-seed{seed}-h100-lr0.000316" for seed in [1, 2, 3, 4]],
    "model 2": [model2_name] + [f"glove-gru-seed{seed}-h128-lr0.001-d0.5" for seed in [1, 2, 3, 4]],
}
for model_name, names in seed_runs.items():
    accuracies = []
    for name in names:
        # the same model and test batches as above
        if model_name == "model 1":
            seed_model = Classifier(1200, 2, inner_dim=model1_inner_dim)
            loader = test_loader1
        else:
            seed_model = RNNClassifier(embedding_matrix, rnn_type=model2_rnn_type, dropout=model2_dropout)
            loader = test_loader2
        seed_model.load_state_dict(torch.load(f"checkpoints/{name}.pth", map_location="cpu", weights_only=True))
        test_loss, test_acc, _ = evaluate(seed_model, loader)
        accuracies.append(test_acc)
    # ddof=1: sample standard deviation of the five runs
    print(f"{model_name}: test accuracy {np.mean(accuracies):.2%} (sd {np.std(accuracies, ddof=1):.2%}, seeds 42, 1, 2, 3, 4: {[round(v * 100, 2) for v in accuracies]})")


## Separate comparison: per-choice scoring
Model 1 with per-choice scoring against the chosen model 1, on the same test examples and with the same exact McNemar test as on the validation set in `04b_preprocessing_experiments.ipynb`. This reports whether the validation gain holds on the test set. It is reported whatever the result, and the selected models do not change.

In [ ]:
from scipy.stats import binomtest

# the predictions of the section "Test results"; True where the prediction is right
correct_chosen = test_predictions[model1_name] == y_test
correct_pc = test_predictions[per_choice_name] == y_test
# & = both True, ~ = not
only_pc = (correct_pc & ~correct_chosen).sum().item()
only_chosen = (~correct_pc & correct_chosen).sum().item()
print("only per-choice right:", only_pc)
print("only chosen model 1 right:", only_chosen)
# exact McNemar test: binomial test on the examples that only one of the two gets right
print(f"p = {binomtest(only_pc, only_pc + only_chosen, 0.5).pvalue:.4f}")

## Errors
Which test examples does each model get right? Count the four combinations (both right, only one of them right, both wrong), compare the two models with the exact McNemar test as in `04_model2_rnn.ipynb`, and look at the first three test examples that model 2 gets wrong. The texts come from the dataset itself (loaded as on slide 9 of the course slides "Course Project HS26"); the rows of the saved test data are in the same order.

In [ ]:
from datasets import load_dataset
from scipy.stats import binomtest

# the test texts, to look at the examples
test = load_dataset("amarfurt/art-nlp-hs26", split="test")

# predicted class (0 = h1, 1 = h2) of every test example, from the section "Test results"
predictions1 = test_predictions[model1_name]
predictions2 = test_predictions[model2_name]
# True where the prediction is right
correct1 = predictions1 == y_test
correct2 = predictions2 == y_test
# & = both True, ~ = not
only_model1 = (correct1 & ~correct2).sum().item()
only_model2 = (~correct1 & correct2).sum().item()
print("both right:", (correct1 & correct2).sum().item())
print("only model 1 right:", only_model1)
print("only model 2 right:", only_model2)
print("both wrong:", (~correct1 & ~correct2).sum().item())
# exact McNemar test: binomial test on the examples that only one model gets right
print(f"p = {binomtest(only_model1, only_model1 + only_model2, 0.5).pvalue:.4f}")

In [ ]:
# the first 3 test examples that model 2 gets wrong (positions where correct2 is False)
for i in torch.nonzero(~correct2).flatten()[:3].tolist():
    example = test[i]
    print("O1:", example["observation_1"])
    print("O2:", example["observation_2"])
    print("h1:", example["hypothesis_1"])
    print("h2:", example["hypothesis_2"])
    # label 1 or 2; predictions are 0 or 1, so + 1
    print(f"correct: h{example['label']}, model 1: h{predictions1[i].item() + 1}, model 2: h{predictions2[i].item() + 1}")
    print()

## Sources
Course materials: NLP HS26 by A. Marfurt, Lucerne University of Applied Sciences and Arts (HSLU), 2026. Full references in README.md.
- Test set only for the final comparison, no tuning on it: course slides "Course Project HS26", slide 10.
- Loading the features: `np.load` as in the NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.savez.html; tensors with `torch.tensor` as in the course exercise "PyTorch" (Week 1); check of the labels with the `assert` statement, Python documentation, https://docs.python.org/3/reference/simple_stmts.html#the-assert-statement, and `torch.equal`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.equal.html.
- Models and evaluation: copied from `03_model1_embeddings.ipynb` and `04_model2_rnn.ipynb` (see the sources there); loading the weights with `load_state_dict(torch.load(..., weights_only=True))` as in the PyTorch tutorial "Save and Load the Model", https://docs.pytorch.org/tutorials/beginner/basics/saveloadrun_tutorial.html.
- Predictions returned by `evaluate()`: an addition of this notebook, so that all results use the same predictions.
- Separate comparison: `PerChoiceClassifier` and the run from `04b_preprocessing_experiments.ipynb` (see the sources there, including slides 25 and 26 of the course slides "Project Discussions" (Week 5) and Bhagavatula et al. (2020), Appendix A.4); exact McNemar test with `scipy.stats.binomtest` as in `04_model2_rnn.ipynb`.
- Errors: loading the dataset as on slide 9 of the course slides "Course Project HS26"; `&` and `~` on True/False tensors (logical AND and NOT) from the PyTorch documentation `torch.bitwise_and` and `torch.bitwise_not`, https://docs.pytorch.org/docs/stable/generated/torch.bitwise_and.html and https://docs.pytorch.org/docs/stable/generated/torch.bitwise_not.html; positions of the wrong predictions with `torch.nonzero`, https://docs.pytorch.org/docs/stable/generated/torch.nonzero.html.
- Hypothesis-only diagnostics: the runs and their sources in `04b_preprocessing_experiments.ipynb`; column indexing `[:, ...]` as in the course exercise "PyTorch" (Week 1); `num_texts` from `04b_preprocessing_experiments.ipynb`; adversarially filtered test set: see "Data" in `README.md`.
- Comparison of the two models: exact McNemar test (McNemar, 1947) with `scipy.stats.binomtest` as in `04_model2_rnn.ipynb` (see the sources there).
- Checkpoints saved on a GPU: `map_location` of `torch.load`, PyTorch documentation, https://docs.pytorch.org/docs/stable/generated/torch.load.html.
- Tools: Claude (Anthropic), ChatGPT (OpenAI) and Gemini (Google), see "Use of AI" in README.md; all code reviewed and understood by me.
- Other seeds: reporting the average of several runs from the course slides "Project Discussions" (Week 7), slide 42; `np.std` with `ddof`, NumPy documentation, https://numpy.org/doc/stable/reference/generated/numpy.std.html; `round()`, Python documentation, https://docs.python.org/3/library/functions.html.
- Setup on Colab: `drive.mount`, Google Colab, https://colab.research.google.com/; `sys.modules`, https://docs.python.org/3/library/sys.html#sys.modules, and `os.chdir`, https://docs.python.org/3/library/os.html#os.chdir, Python documentation; `%pip`, IPython documentation, https://ipython.readthedocs.io/en/stable/interactive/magics.html.